# Day 04 下午项目作业：电商用户行为数据清洗
**学生：** 金睿博  
**学号：** 24012434  
**日期：** 2026年7月  

---

## 项目背景
本节课的目标是将上午学到的各种数据清洗方法（缺失值处理、类别标准化、异常值检查等）整合成一个完整的、可复用的清洗流程。

最终需要在 output/day04_project/ 文件夹中生成4个交付文件：
1. ecommerce_customer_cleaned.csv（清洗后的数据）
2. data_quality_before.csv（清洗前质量报告）
3. data_quality_after.csv（清洗后质量报告）
4. cleaning_log.csv（处理日志）

**项目原则：**
- 原始数据只读，不覆盖
- 清洗函数接收 DataFrame，返回清洗结果和处理日志
- 不用 Churn（目标变量）做分组填充，避免数据泄露
- 候选异常值先记录判断，不盲目删除

---
## 任务1：项目初始化与数据理解

首先需要读取数据并回答三个问题：
1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

# 读取数据（注意这个Excel有两个sheet，数据在 E Comm 表）
DATA_PATH = Path(r"C:\Users\resch\Downloads\E Commerce Dataset.xlsx")
raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"数据形状：{raw_df.shape}")
print(f"字段列表：{raw_df.columns.tolist()}")
raw_df.head()

In [ ]:
# 任务1 回答
#
# 问题1：每条记录代表什么？
# 答：每条记录代表一位电商平台的用户。数据共有 5630 条记录，
# 每行记录了一位用户的各种行为信息（使用时长、下单次数、登录设备等）。
#
# 问题2：目标变量是哪一列？
# 答：目标变量是 Churn（是否流失），取值为 0 或 1，
# 1 表示该用户已流失，0 表示仍在活跃使用平台。
#
# 问题3：为什么 CustomerID 不应作为普通数值参与分析？
# 答：CustomerID 是用户的唯一标识符，从 50001 开始编号，
# 它本身没有数值意义（不能做加减乘除），如果当成连续数值参与建模，
# 模型会被编号大小误导，得出没有业务意义的结论。

---
## 任务2：数据质量审计

在做任何处理之前，先全面检查数据质量。需要回答：
- 各字段的缺失情况如何？
- 有没有重复记录？有没有重复用户？
- 各类别字段的取值是否一致？

In [ ]:
# 构建数据质量报告函数（后面清洗完还要用）
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().mean() * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

quality_before = build_quality_report(raw_df)
print("=== 清洗前数据质量报告 ===")
display(quality_before[quality_before["缺失数量"] > 0])  # 只看有缺失的字段

In [ ]:
# 初始审计
print("=== 重复值检查 ===")
print(f"完全重复行数：{raw_df.duplicated().sum()}")
print(f"CustomerID 重复数：{raw_df["CustomerID"].duplicated().sum()}")

print("\n=== 目标变量 Churn ===")
print(raw_df["Churn"].value_counts())
print(f"流失率：{raw_df["Churn"].mean() * 100:.2f}%")

print("\n=== 类别字段取值检查 ===")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}：")
    print(raw_df[col].value_counts())

**审计发现：**
- 7个数值字段存在缺失值，缺失率在 4.5%~5.5% 之间
- 没有重复行，也没有重复用户
- PreferredLoginDevice 里 Phone 和 Mobile Phone 其实是同一个东西
- PreferredPaymentMode 里 COD = Cash on Delivery，CC = Credit Card
- PreferedOrderCat 里 Mobile 和 Mobile Phone 也是同一品类

---
## 任务3：编写清洗函数

根据上面的审计结果，确定以下清洗规则：

| 问题 | 处理方式 | 理由 |
|------|----------|------|
| 数值字段缺失 | 总体中位数填充 | 中位数比均值更稳健，不会被极端值拉偏 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务含义 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一支付方式 |
| CC / Credit Card | 统一为 Credit Card | 同一支付方式 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一品类 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不提供额外信息 |

注意：不能用 Churn 分组填充，否则等于把目标变量的信息泄露给特征。

运行清洗函数后，查看处理日志确认每一步的影响。

In [ ]:
# 定义清洗规则常量
NUMERIC_MISSING_COLS = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "CouponUsed",
    "OrderCount", "DaySinceLastOrder"
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {"Phone": "Mobile Phone"},
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {"Mobile": "Mobile Phone"}
}

In [ ]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    接收原始 DataFrame，返回清洗后的 DataFrame 和处理日志。
    """
    cleaned_df = data.copy()  # 不修改原始数据
    n_before = len(cleaned_df)
    logs = []

    # 第一步：删除完全重复行
    dup_count = cleaned_df.duplicated().sum()
    if dup_count > 0:
        cleaned_df = cleaned_df.drop_duplicates()
        logs.append({
            "处理步骤": "删除重复行",
            "处理规则": "drop_duplicates",
            "处理前记录数": n_before,
            "处理后记录数": len(cleaned_df),
            "影响记录数": int(dup_count)
        })

    # 第二步：数值字段中位数填充
    for col in NUMERIC_MISSING_COLS:
        na_count = cleaned_df[col].isna().sum()
        if na_count > 0:
            cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())
            logs.append({
                "处理步骤": "缺失值填补",
                "处理规则": f"{col} 中位数填补",
                "处理前记录数": n_before,
                "处理后记录数": n_before,
                "影响记录数": int(na_count)
            })

    # 第三步：类别标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            affected = (cleaned_df[col] == old_val).sum()
            if affected > 0:
                cleaned_df[col] = cleaned_df[col].replace(old_val, new_val)
                logs.append({
                    "处理步骤": "类别标准化",
                    "处理规则": f"{col}: {old_val} -> {new_val}",
                    "处理前记录数": n_before,
                    "处理后记录数": n_before,
                    "影响记录数": int(affected)
                })

    # 第四步：数据类型转换
    cleaned_df["Churn"] = cleaned_df["Churn"].astype(int)
    cleaned_df["Complain"] = cleaned_df["Complain"].astype(int)
    logs.append({
        "处理步骤": "数据类型转换",
        "处理规则": "Churn/Complain 转 int",
        "处理前记录数": n_before,
        "处理后记录数": n_before,
        "影响记录数": 0
    })

    cleaning_log = pd.DataFrame(logs)
    return cleaned_df, cleaning_log

In [ ]:
# 运行清洗函数
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

print(f"清洗前记录数：{len(raw_df)}")
print(f"清洗后记录数：{len(cleaned_df)}")
print("\n=== 处理日志 ===")
display(cleaning_log)
print("\n=== 清洗后数据预览 ===")
display(cleaned_df.head())

---
## 任务4：新增分析字段 + 异常值检查

为了第五天的分析做准备，需要新增两个字段：
- **TenureGroup**：用户使用时长分层（0-1年、1-5年、5-10年等）
- **IsMobileLogin**：是否使用移动端登录（1=是，0=否）

然后用 IQR 方法检查以下字段的候选异常值：
- WarehouseToHome（仓库到家距离）
- OrderCount（订单数）
- CashbackAmount（返现金额）

注意：IQR 筛选出来的是"候选"异常值，只记录不删除。

In [ ]:
# 新增 TenureGroup
tenure_bins = [-1, 1, 5, 10, 15, 100]
tenure_labels = ["0-1年", "1-5年", "5-10年", "10-15年", "15年以上"]
cleaned_df["TenureGroup"] = pd.cut(
    cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels
)

# 新增 IsMobileLogin
cleaned_df["IsMobileLogin"] = (
    cleaned_df["PreferredLoginDevice"] == "Mobile Phone"
).astype(int)

print("TenureGroup 分布：")
print(cleaned_df["TenureGroup"].value_counts().sort_index())
print(f"\n移动端登录用户占比：{cleaned_df["IsMobileLogin"].mean() * 100:.1f}%")

In [ ]:
# IQR 候选异常值检查
def iqr_outlier_summary(series):
    """用 IQR 方法识别候选异常值。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((series < lower) | (series > upper)).sum())
    return {"Q1": q1, "Q3": q3, "下限": lower, "上限": upper,
            "候选异常值数量": outlier_count}

outlier_fields = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_rows = []
for col in outlier_fields:
    summary = iqr_outlier_summary(cleaned_df[col])
    summary["字段"] = col
    outlier_rows.append(summary)
outlier_report = pd.DataFrame(outlier_rows)[["字段", "Q1", "Q3", "下限", "上限", "候选异常值数量"]]

print("=== IQR 候选异常值报告 ===")
display(outlier_report)
print("\n说明：以上字段筛选出来的是候选异常值，需结合业务判断是否处理。")

In [ ]:
# 业务规则检查
business_checks = {
    "使用时长(Tenure) < 0": (cleaned_df["Tenure"] < 0).sum(),
    "仓库距离(WarehouseToHome) < 0": (cleaned_df["WarehouseToHome"] < 0).sum(),
    "订单数(OrderCount) <= 0": (cleaned_df["OrderCount"] <= 0).sum(),
    "返现金额(CashbackAmount) < 0": (cleaned_df["CashbackAmount"] < 0).sum()
}

business_rule_report = pd.DataFrame(
    list(business_checks.items()),
    columns=["规则", "不合规记录数"]
)
display(business_rule_report)

total_violations = business_rule_report["不合规记录数"].sum()
if total_violations == 0:
    print("结论：所有业务规则检查通过，未发现不合规记录。")
else:
    print(f"结论：共发现 {int(total_violations)} 条不合规记录，需进一步核查。")

---
## 任务5：验收与导出

清洗完成后，需要：
1. 生成清洗后的质量报告，跟清洗前对比
2. 做验收检查（确保所有处理都生效了）
3. 导出4个交付文件

In [ ]:
# 清洗前后对比
quality_after = build_quality_report(cleaned_df)
comparison = pd.DataFrame({
    "清洗前缺失数": quality_before["缺失数量"],
    "清洗后缺失数": quality_after["缺失数量"]
})
print("=== 清洗前后缺失值对比 ===")
display(comparison[comparison["清洗前缺失数"] > 0])

In [ ]:
# 验收检查
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍有缺失"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备未统一"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式COD未统一"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式CC未统一"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "新增字段缺失"
print("✅ 所有验收检查通过")

In [ ]:
# 导出交付文件
OUTPUT_DIR = Path("output/day04_project")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

print("=== 交付文件 ===")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f}")
print("\n全部完成！")

---
## 项目复盘

In [ ]:
# 1. 本项目发现了哪些数据质量问题？
#
# 数据中共有7个数值字段存在缺失值（Tenure、WarehouseToHome、HourSpendOnApp、
# OrderAmountHikeFromlastYear、CouponUsed、OrderCount、DaySinceLastOrder），
# 缺失率在 4.5%~5.5% 之间。登录设备和支付方式字段存在类别取值不一致的问题，
# 比如 Phone 和 Mobile Phone 实际是同一设备，COD 和 Cash on Delivery 是同一支付方式。
# 经检查无重复行，也没有重复用户。
#
# 2. 对缺失值、类别不一致、候选异常值分别采取了什么策略？
#
# 缺失值：采用总体中位数填充。不选均值是因为中位数不受极端值影响，更稳健；
# 不用 0 填充是因为"缺失"不等于"没有"。同时没有按 Churn 分组填充，
# 避免将目标变量的信息泄露到特征中。
#
# 类别不一致：通过字典映射统一写法，比如把 Phone 映射为 Mobile Phone、
# COD 映射为 Cash on Delivery。合并前确认了业务逻辑上的等价性。
#
# 候选异常值：用 IQR 方法识别并记录数量，但不自动删除。因为 IQR 筛出的
# 只是统计意义上的极端值，是否真的是错误数据需要业务判断。
#
# 3. 为什么清洗后的数据可以作为第五天分析的输入？
#
# 清洗后所有数值字段缺失值为 0，类别字段取值统一，新增的 TenureGroup
# 和 IsMobileLogin 字段为后续分析提供了分组维度。所有处理步骤都有日志记录，
# 处理过程可追溯、可复现，数据质量满足统计分析和建模的基本要求。
#
# 4. 哪些处理规则仍需要业务人员确认？
#
# 首先是类别合并的合理性：比如 PreferedOrderCat 中 Mobile 和 Mobile Phone
# 是否确实是同一品类，需要业务确认。其次是候选异常值的最终处理：
# IQR 标记出的极端值应该保留、删除还是标记，取决于实际业务场景。
# 最后是中位数填充的分组粒度：当前用的是全表中位数，但如果不同城市等级
# 的仓储距离差异较大，可能需要按 CityTier 分组填充。